# YOLO 3-Stage Training Pipeline
**De tai:** Ung dung hoc sau trong nhan dien phuong tien va uoc tinh mat do giao thong  
**Sinh vien:** Nguyen Huynh — 102220024

---

## Tong quan quy trinh

```
[Giai doan 0] Moi truong + Drive
      |
[Giai doan 1] COCO Pretrained weights  --> yolov8m.pt
      |
[Giai doan 2] Finetune BDD100K         --> stage2_bdd100k/best.pt
      |       (freeze backbone, 30 epochs, lr=0.005)
[Giai doan 3] Finetune Vietnam data    --> stage3_vn_final/best.pt
      |       (full unfreeze, 100 epochs, lr=0.001)
[Giai doan 4] Danh gia + Export ONNX
```

**Truoc khi chay:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload anh va annotation len Google Drive
3. Chay tung cell theo thu tu

---
# GIAI DOAN 0 — Moi truong & Cai dat

In [ ]:
# [0.1] Kiem tra GPU
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    lines = result.stdout.strip().split('\n')
    for line in lines[:12]: print(line)
    print('\nGPU san sang!')
else:
    print('KHONG CO GPU!')
    print('Vao Runtime > Change runtime type > chon T4 GPU')

import psutil
ram = psutil.virtual_memory()
print(f'\nRAM tong: {ram.total/1e9:.1f} GB | Trong: {ram.available/1e9:.1f} GB')

In [ ]:
# [0.2] Cai dat thu vien
%%time
import subprocess, sys

packages = [
    'ultralytics>=8.3.0',
    'albumentations>=1.4.0',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import torch
from ultralytics import YOLO
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print('Cai dat hoan tat!')

In [ ]:
# [0.3] Import thu vien
import os, yaml, shutil, random, time, json
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.notebook import tqdm
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('Import OK!')

In [ ]:
# [0.4] CAU HINH TOAN CUC — chinh sua o day
# ================================================================
CFG = {
    # --- Model ---
    'base_model'   : 'yolov8m.pt',  # yolov8n/s/m/l hoac yolo11m.pt

    # --- Classes ---
    'num_classes'  : 4,
    'class_names'  : ['xe_may', 'o_to', 'xe_tai', 'xe_bus'],

    # --- Google Drive ---
    'drive_root'   : '/content/drive/MyDrive/DATN_Data',

    # --- Local paths ---
    'local_data'   : '/content/data',
    'runs_dir'     : '/content/runs',

    # --- Giai doan 2: Finetune BDD100K ---
    'stage2_epochs'   : 30,
    'stage2_lr0'      : 0.005,
    'stage2_lrf'      : 0.0005,
    'stage2_freeze'   : 10,      # dong bang 10 lop backbone
    'stage2_batch'    : 16,

    # --- Giai doan 3: Finetune VN data ---
    'stage3_epochs'   : 100,
    'stage3_lr0'      : 0.001,
    'stage3_lrf'      : 0.0001,
    'stage3_freeze'   : 0,       # mo toan bo
    'stage3_batch'    : 16,
    'stage3_patience' : 20,      # early stopping

    'img_size'     : 640,
}

print('Cau hinh:')
for k, v in CFG.items():
    print(f'  {k:20s}: {v}')

---
# GIAI DOAN 0 (tiep) — Ket noi Google Drive

> Upload du lieu len Drive theo cau truc duoi day truoc khi chay cell nay:
>
> ```
> DATN_Data/
>   bdd100k/          <- du lieu BDD100K (optional)
>     images/train/
>     images/val/
>     labels/train/
>     labels/val/
>   vn_data/          <- du lieu tu thu thap (bat buoc)
>     images/train/
>     images/val/
>     images/test/
>     labels/train/
>     labels/val/
>     labels/test/
> ```

In [ ]:
# [0.5] Mount Google Drive
from google.colab import drive

print('Dang ket noi Google Drive...')
drive.mount('/content/drive', force_remount=False)

drive_root = Path(CFG['drive_root'])
print(f'\nDrive root: {drive_root}')

# Kiem tra cac thu muc can thiet
checks = {
    'BDD100K train': drive_root / 'bdd100k' / 'images' / 'train',
    'BDD100K val'  : drive_root / 'bdd100k' / 'images' / 'val',
    'VN train'     : drive_root / 'vn_data'  / 'images' / 'train',
    'VN val'       : drive_root / 'vn_data'  / 'images' / 'val',
    'VN test'      : drive_root / 'vn_data'  / 'images' / 'test',
}

print()
all_ok = True
for name, path in checks.items():
    if path.exists():
        n = len(list(path.glob('*.jpg')) + list(path.glob('*.png')))
        print(f'  OK  {name:18s}: {n} anh')
    else:
        print(f'  --  {name:18s}: chua co ({path})')
        if 'VN' in name: all_ok = False

if not all_ok:
    print('\nCHU Y: Chua co du lieu VN. Hay upload truoc khi chay Giai doan 3.')
else:
    print('\nTat ca du lieu san sang!')

In [ ]:
# [0.6] Tao cau truc thu muc neu chua co
drive_root = Path(CFG['drive_root'])

folders = [
    'bdd100k/images/train', 'bdd100k/images/val',
    'bdd100k/labels/train', 'bdd100k/labels/val',
    'vn_data/images/train', 'vn_data/images/val', 'vn_data/images/test',
    'vn_data/labels/train', 'vn_data/labels/val', 'vn_data/labels/test',
    'checkpoints',
    'results',
]
for f in folders:
    (drive_root / f).mkdir(parents=True, exist_ok=True)

print('Cau truc thu muc tren Drive:')
print(f'{drive_root}/')
for f in folders:
    print(f'  {f}/')
print('\nHoan tat tao thu muc!')

---
# GIAI DOAN 0 (tiep) — Tao file dataset.yaml

In [ ]:
# [0.7] Tao dataset.yaml cho BDD100K (Giai doan 2)
import yaml

BDD_YAML = '/content/bdd100k.yaml'
bdd_cfg = {
    'path' : str(drive_root / 'bdd100k'),
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : CFG['num_classes'],
    'names': CFG['class_names'],
}
with open(BDD_YAML, 'w') as f:
    yaml.dump(bdd_cfg, f, default_flow_style=False)

# Tao dataset.yaml cho VN data (Giai doan 3)
VN_YAML = '/content/vn_traffic.yaml'
vn_cfg = {
    'path' : str(drive_root / 'vn_data'),
    'train': 'images/train',
    'val'  : 'images/val',
    'test' : 'images/test',
    'nc'   : CFG['num_classes'],
    'names': CFG['class_names'],
}
with open(VN_YAML, 'w') as f:
    yaml.dump(vn_cfg, f, default_flow_style=False)

print('BDD100K yaml:')
with open(BDD_YAML) as f: print(f.read())
print('VN Traffic yaml:')
with open(VN_YAML) as f: print(f.read())

In [ ]:
# [0.8] Ham tien ich: kiem tra dataset
def verify_dataset(yaml_path, title='Dataset'):
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    data_path = Path(cfg['path'])
    nc = cfg['nc']
    names = cfg['names']
    print(f'\n=== {title} ===')
    print(f'Path: {data_path}')
    splits = ['train','val'] + (['test'] if 'test' in cfg else [])
    total_ok = True
    for split in splits:
        img_dir = data_path / cfg.get(split, f'images/{split}')
        lbl_dir = Path(str(img_dir).replace('images','labels'))
        if not img_dir.exists():
            print(f'  {split:5s}: KHONG TIM THAY {img_dir}'); total_ok=False; continue
        imgs = sorted(f for f in img_dir.iterdir()
                      if f.suffix.lower() in {'.jpg','.jpeg','.png'})
        cc = [0]*nc; miss=0
        for img in imgs:
            lbl = lbl_dir/(img.stem+'.txt')
            if not lbl.exists(): miss+=1; continue
            for line in lbl.read_text().strip().splitlines():
                p=line.strip().split()
                if len(p)==5:
                    cid=int(p[0])
                    if 0<=cid<nc: cc[cid]+=1
        tb=sum(cc)
        cs=' | '.join(f'{names[i]}:{cc[i]}' for i in range(nc))
        warn=' [MISS LABELS!]' if miss else ''
        print(f'  {split:5s}: {len(imgs):5d} anh | {tb:6d} bbox | {cs}{warn}')
    return total_ok

print('Ham verify_dataset da san sang.')
print('Chay: verify_dataset(BDD_YAML, "BDD100K") hoac verify_dataset(VN_YAML, "VN Data")')

---
# GIAI DOAN 1 — COCO Pretrained Weights

> Khong can train lai. Ultralytics da lam san.
> Cell nay chi tai weights va kiem tra.

In [ ]:
# [1.1] Tai COCO pretrained weights
print('='*55)
print('GIAI DOAN 1 — COCO Pretrained')
print('='*55)
print(f'Model: {CFG["base_model"]}')
print('Dang tai weights tu Ultralytics Hub...')

model_g1 = YOLO(CFG['base_model'])
model_g1.info(verbose=False)

# So tham so
params = sum(p.numel() for p in model_g1.model.parameters())
print(f'\nSo tham so: {params/1e6:.1f}M')
print(f'Device    : {DEVICE}')
print()
print('GIAI DOAN 1 HOAN TAT!')
print(f'Weights: {CFG["base_model"]} (COCO pretrained)')

# Luu ten file cho giai doan tiep theo
STAGE1_WEIGHTS = CFG['base_model']
print(f'\n-> Weights dua vao Giai doan 2: {STAGE1_WEIGHTS}')

In [ ]:
# [1.2] Kiem tra kha nang nhan biet phuong tien tren anh thu (optional)
# Chay thu inference mot lan de xem model COCO hoat dong nhu the nao
# truoc khi finetune

import numpy as np

# Tao anh ngau nhien de test (thay bang anh that neu co)
dummy = np.random.randint(50, 200, (640,640,3), dtype=np.uint8)

result = model_g1.predict(dummy, conf=0.25, verbose=False)[0]
print(f'So doi tuong phat hien (anh ngau nhien): {len(result.boxes)}')
print('Model COCO san sang. Bat dau Giai doan 2...')

---
# GIAI DOAN 2 — Finetune tren BDD100K

**Muc tieu:** Thu hep domain gap giua COCO (da dang) va anh giao thong thuc te.  
**Chien luoc:** Dong bang backbone (freeze=10), chi train detection head.  
**Thoi gian uoc tinh:** 30 epochs ~ 40-60 phut tren T4.

> **Neu khong co BDD100K:** Bo qua cell [2.1] den [2.3],
> dat `STAGE2_WEIGHTS = STAGE1_WEIGHTS` va chay tiep Giai doan 3.

In [ ]:
# [2.1] Kiem tra du lieu BDD100K
print('='*55)
print('GIAI DOAN 2 — Finetune BDD100K')
print('='*55)

bdd_ok = verify_dataset(BDD_YAML, 'BDD100K')
if bdd_ok:
    print('\nDu lieu BDD100K hop le. Bat dau train...')
else:
    print('\nCHU Y: Du lieu BDD100K chua day du!')
    print('Neu muon bo qua Giai doan 2:')
    print('  STAGE2_WEIGHTS = STAGE1_WEIGHTS')
    print('  Roi chay thang Giai doan 3.')

In [ ]:
# [2.2] Cau hinh Giai doan 2
STAGE2_CFG = {
    'data'           : BDD_YAML,
    'epochs'         : CFG['stage2_epochs'],
    'batch'          : CFG['stage2_batch'],
    'imgsz'          : CFG['img_size'],
    'device'         : DEVICE,
    'workers'        : 2,
    'seed'           : SEED,

    # Optimizer
    'lr0'            : CFG['stage2_lr0'],   # 0.005 (thap hon scratch)
    'lrf'            : CFG['stage2_lrf'],   # 0.0005
    'momentum'       : 0.937,
    'weight_decay'   : 0.0005,

    # Warmup
    'warmup_epochs'  : 3.0,
    'warmup_momentum': 0.8,

    # Dong bang backbone — chi train detection head
    'freeze'         : CFG['stage2_freeze'],  # dong bang 10 lop dau

    # Augmentation nhe
    'mosaic'         : 1.0,
    'mixup'          : 0.0,   # tat mixup giai doan nay
    'fliplr'         : 0.5,
    'flipud'         : 0.0,
    'degrees'        : 3.0,   # xoay nhe
    'scale'          : 0.3,
    'hsv_v'          : 0.4,
    'hsv_s'          : 0.7,
    'hsv_h'          : 0.015,

    # Save
    'project'        : CFG['runs_dir'],
    'name'           : 'stage2_bdd100k',
    'save'           : True,
    'save_period'    : 10,
    'exist_ok'       : True,
    'val'            : True,
    'plots'          : True,
    'amp'            : True,
    'verbose'        : True,
}

print('Cau hinh Giai doan 2:')
for k, v in STAGE2_CFG.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# [2.3] TRAIN — Giai doan 2
print('BAT DAU GIAI DOAN 2...')
print(f'Model   : {STAGE1_WEIGHTS}')
print(f'Dataset : BDD100K (4 class)')
print(f'Epochs  : {STAGE2_CFG["epochs"]}')
print(f'Freeze  : {STAGE2_CFG["freeze"]} lop backbone')
print(f'LR      : {STAGE2_CFG["lr0"]}')
print()

model_g2 = YOLO(STAGE1_WEIGHTS)

t0 = __import__('time').time()
results_g2 = model_g2.train(**STAGE2_CFG)
elapsed = (__import__('time').time() - t0) / 60

STAGE2_WEIGHTS = f"{CFG['runs_dir']}/stage2_bdd100k/weights/best.pt"

print()
print('='*55)
print(f'GIAI DOAN 2 HOAN TAT! ({elapsed:.1f} phut)')
print(f'Best weights: {STAGE2_WEIGHTS}')
print(f'-> Weights dua vao Giai doan 3: {STAGE2_WEIGHTS}')

In [ ]:
# [2.4] Xem ket qua Giai doan 2
save_dir = Path(CFG['runs_dir']) / 'stage2_bdd100k'
plots = [
    ('results.png',  'Loss & mAP theo epoch'),
    ('PR_curve.png', 'Precision-Recall Curve'),
]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (fname, title) in zip(axes, plots):
    fpath = save_dir / fname
    if fpath.exists():
        ax.imshow(plt.imread(str(fpath)))
        ax.set_title(f'[Giai doan 2] {title}', fontsize=11)
    else:
        ax.text(0.5,0.5,f'Chua co: {fname}',ha='center',va='center',transform=ax.transAxes)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f"{CFG['runs_dir']}/stage2_results_preview.png", dpi=100, bbox_inches='tight')
plt.show()

# Luu checkpoint len Drive
drive_ckpt = drive_root / 'checkpoints'
drive_ckpt.mkdir(parents=True, exist_ok=True)
if Path(STAGE2_WEIGHTS).exists():
    shutil.copy2(STAGE2_WEIGHTS, drive_ckpt / 'stage2_best.pt')
    print(f'Saved to Drive: {drive_ckpt}/stage2_best.pt')

---
# GIAI DOAN 3 — Finetune tren data Viet Nam

**Muc tieu:** Hoc dac thu giao thong VN: xe may mat do cao, khong phan lan,
goc camera giam sat co dinh.

**Chien luoc:** Mo toan bo (freeze=0), LR rat thap (0.001) de khong quen BDD100K.

**Thoi gian uoc tinh:** 100 epochs ~ 1.5–2.5 gio tren T4.

> Neu Giai doan 2 bi bo qua, set:
> `STAGE2_WEIGHTS = STAGE1_WEIGHTS`  (train thang tu COCO)

In [ ]:
# [3.0] Xac dinh weights dau vao Giai doan 3
# Mac dinh dung ket qua Giai doan 2.
# Neu da bo qua G2, bo comment dong duoi:
# STAGE2_WEIGHTS = STAGE1_WEIGHTS

# Neu can load tu Drive (khi Colab bi reset):
# STAGE2_WEIGHTS = str(drive_root / 'checkpoints' / 'stage2_best.pt')

print(f'Weights dau vao Giai doan 3: {STAGE2_WEIGHTS}')
print(f'File ton tai: {Path(STAGE2_WEIGHTS).exists()}')

In [ ]:
# [3.1] Kiem tra du lieu VN
print('='*55)
print('GIAI DOAN 3 — Finetune VN Data')
print('='*55)

vn_ok = verify_dataset(VN_YAML, 'Vietnam Traffic Data')
if not vn_ok:
    print('\nLOI: Thieu du lieu VN!')
    print('Upload anh + annotation vao:')
    print(f'  {drive_root}/vn_data/images/train/')
    print(f'  {drive_root}/vn_data/labels/train/')
    raise RuntimeError('Thieu du lieu VN. Hay upload truoc.')

In [ ]:
# [3.2] Cau hinh Giai doan 3
STAGE3_CFG = {
    'data'           : VN_YAML,
    'epochs'         : CFG['stage3_epochs'],
    'patience'       : CFG['stage3_patience'],  # early stopping
    'batch'          : CFG['stage3_batch'],
    'imgsz'          : CFG['img_size'],
    'device'         : DEVICE,
    'workers'        : 2,
    'seed'           : SEED,

    # Optimizer — LR rat thap de khong quen BDD100K
    'lr0'            : CFG['stage3_lr0'],   # 0.001
    'lrf'            : CFG['stage3_lrf'],   # 0.0001
    'momentum'       : 0.937,
    'weight_decay'   : 0.0005,

    # Warmup
    'warmup_epochs'  : 3.0,
    'warmup_momentum': 0.8,

    # Mo toan bo — train ca backbone + neck + head
    'freeze'         : CFG['stage3_freeze'],  # 0 = khong dong bang gi

    # Augmentation manh hon de bu du lieu it
    'mosaic'         : 1.0,
    'mixup'          : 0.1,   # bat mixup
    'fliplr'         : 0.5,
    'flipud'         : 0.0,   # khong lat doc (giao thong)
    'degrees'        : 5.0,   # xoay nhieu hon G2
    'scale'          : 0.5,
    'perspective'    : 0.0,
    'hsv_h'          : 0.015,
    'hsv_s'          : 0.7,
    'hsv_v'          : 0.4,
    'translate'      : 0.1,
    'close_mosaic'   : 10,    # tat mosaic 10 epoch cuoi

    # Save
    'project'        : CFG['runs_dir'],
    'name'           : 'stage3_vn_final',
    'save'           : True,
    'save_period'    : 10,
    'exist_ok'       : True,
    'val'            : True,
    'plots'          : True,
    'amp'            : True,
    'verbose'        : True,
}

print('Cau hinh Giai doan 3:')
for k, v in STAGE3_CFG.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# [3.3] TRAIN — Giai doan 3 (QUAN TRONG NHAT)
print('BAT DAU GIAI DOAN 3 (giai doan quan trong nhat de tai)...')
print(f'Model   : {STAGE2_WEIGHTS}')
print(f'Dataset : Vietnam Traffic (tu thu thap)')
print(f'Epochs  : {STAGE3_CFG["epochs"]} (early stop after {STAGE3_CFG["patience"]} no-improve)')
print(f'Freeze  : {STAGE3_CFG["freeze"]} (mo toan bo)')
print(f'LR      : {STAGE3_CFG["lr0"]} (rat thap de khong quen BDD100K)')
print()

model_g3 = YOLO(STAGE2_WEIGHTS)

t0 = __import__('time').time()
results_g3 = model_g3.train(**STAGE3_CFG)
elapsed = (__import__('time').time() - t0) / 60

STAGE3_WEIGHTS = f"{CFG['runs_dir']}/stage3_vn_final/weights/best.pt"
STAGE3_LAST    = f"{CFG['runs_dir']}/stage3_vn_final/weights/last.pt"

print()
print('='*55)
print(f'GIAI DOAN 3 HOAN TAT! ({elapsed:.1f} phut)')
print(f'Best weights: {STAGE3_WEIGHTS}')
print(f'Last weights: {STAGE3_LAST}')

# Luu ngay len Drive
drive_ckpt = drive_root / 'checkpoints'
shutil.copy2(STAGE3_WEIGHTS, drive_ckpt / 'stage3_best.pt')
shutil.copy2(STAGE3_LAST,    drive_ckpt / 'stage3_last.pt')
print(f'\nDa luu len Drive: {drive_ckpt}')

In [ ]:
# [3.4] Xem ket qua Giai doan 3
save_dir = Path(CFG['runs_dir']) / 'stage3_vn_final'
plots = [
    ('results.png',          'Loss & mAP'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('PR_curve.png',         'PR Curve'),
    ('F1_curve.png',         'F1 Curve'),
]
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, (fname, title) in zip(axes.flatten(), plots):
    fpath = save_dir / fname
    if fpath.exists():
        ax.imshow(plt.imread(str(fpath)))
        ax.set_title(f'[Giai doan 3] {title}', fontsize=11)
    else:
        ax.text(0.5,0.5,f'Chua co: {fname}',ha='center',va='center',transform=ax.transAxes)
    ax.axis('off')
plt.tight_layout()
plt.savefig(str(drive_root/'results'/'stage3_overview.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Da luu overview len Drive/results/')

---
# GIAI DOAN 4 — Danh gia & Export

Danh gia tren tap test, so sanh baseline, va export sang ONNX.

In [ ]:
# [4.0] Load best model
# Neu Colab bi reset, load tu Drive:
# STAGE3_WEIGHTS = str(drive_root / 'checkpoints' / 'stage3_best.pt')

print('='*55)
print('GIAI DOAN 4 — Danh gia & Export')
print('='*55)
print(f'Loading: {STAGE3_WEIGHTS}')

best_model = YOLO(STAGE3_WEIGHTS)
print('Model loaded OK!')

In [ ]:
# [4.1] Danh gia tren TEST SET
print('Danh gia tren TEST SET...')
print('-'*50)

metrics = best_model.val(
    data    = VN_YAML,
    split   = 'test',
    imgsz   = CFG['img_size'],
    conf    = 0.25,
    iou     = 0.6,
    plots   = True,
    verbose = False,
)

map50   = metrics.box.map50
map5095 = metrics.box.map
prec    = metrics.box.mp
rec     = metrics.box.mr

print(f'mAP@0.5      : {map50:.4f}  {"OK >=0.85" if map50>=0.85 else "CHUA DAT <0.85"}')
print(f'mAP@0.5:0.95 : {map5095:.4f}')
print(f'Precision    : {prec:.4f}')
print(f'Recall       : {rec:.4f}')
print()
print('Per-class mAP@0.5:')
for i,(name,ap) in enumerate(zip(CFG['class_names'], metrics.box.ap50)):
    bar = '#'*int(ap*20)
    ok = 'OK' if ap>=0.80 else '--'
    print(f'  [{ok}] {name:8s}: {ap:.4f} |{bar:<20}|')

# Luu ket qua
results_dict = {
    'mAP@0.5': round(float(map50),4),
    'mAP@0.5:0.95': round(float(map5095),4),
    'Precision': round(float(prec),4),
    'Recall': round(float(rec),4),
    'per_class': {name: round(float(ap),4)
                  for name,ap in zip(CFG['class_names'], metrics.box.ap50)},
}
with open(str(drive_root/'results'/'final_metrics.json'),'w') as f:
    json.dump(results_dict, f, indent=2)
print('\nDa luu metrics: Drive/results/final_metrics.json')

In [ ]:
# [4.2] Do FPS tren GPU
import time

print('Do toc do inference...')
dummy_img = np.random.randint(0,255,(640,640,3),dtype=np.uint8)

# Warm up
for _ in range(5):
    best_model.predict(dummy_img, verbose=False)

# Do chinh xac
N_RUNS = 100
t0 = time.perf_counter()
for _ in range(N_RUNS):
    best_model.predict(dummy_img, imgsz=640, verbose=False)
elapsed = time.perf_counter() - t0

fps     = N_RUNS / elapsed
lat_ms  = elapsed / N_RUNS * 1000

print(f'FPS         : {fps:.1f}  {"OK >=15" if fps>=15 else "CHUA DAT <15"}')
print(f'Latency     : {lat_ms:.1f} ms/frame')
print(f'GPU         : {torch.cuda.get_device_name(0)}')

results_dict['fps']         = round(fps,1)
results_dict['latency_ms']  = round(lat_ms,1)
with open(str(drive_root/'results'/'final_metrics.json'),'w') as f:
    json.dump(results_dict, f, indent=2)
print('\nDa cap nhat FPS vao final_metrics.json')

In [ ]:
# [4.3] So sanh baseline 3 model (cho bao cao Chuong 4)
import pandas as pd

print('So sanh baseline...')
print('(Moi model train 20 epochs nhanh de so sanh)')

COMPARE = [
    ('yolov8n.pt', 'YOLOv8n (scratch->VN)'),
    ('yolov8s.pt', 'YOLOv8s (scratch->VN)'),
    (CFG['base_model'], 'YOLOv8m (scratch->VN)'),
]
# Them ket qua 3-stage cua ta
compare_rows = []

for wt, label in COMPARE:
    print(f'  Training baseline: {label}...')
    m = YOLO(wt)
    m.train(
        data=VN_YAML, epochs=20, batch=CFG['stage3_batch'],
        imgsz=CFG['img_size'], device=DEVICE,
        project=CFG['runs_dir'],
        name=f'cmp_{Path(wt).stem}',
        exist_ok=True, verbose=False, plots=False, amp=True, workers=2,
    )
    vm = m.val(data=VN_YAML, split='test', verbose=False)
    p  = sum(x.numel() for x in m.model.parameters())/1e6

    # Do FPS
    t0=time.perf_counter()
    for _ in range(30): m.predict(dummy_img, verbose=False)
    fps_b = 30/(time.perf_counter()-t0)

    compare_rows.append({
        'Phuong phap': label,
        'mAP@0.5'    : round(float(vm.box.map50),4),
        'mAP@0.5:0.95': round(float(vm.box.map),4),
        'Precision'  : round(float(vm.box.mp),4),
        'Recall'     : round(float(vm.box.mr),4),
        'FPS'        : round(fps_b,1),
        'Params(M)'  : round(p,1),
    })

# Them ket qua pipeline 3 giai doan
compare_rows.append({
    'Phuong phap': f'YOLOv8m (COCO->BDD->VN) [DE TAI]',
    'mAP@0.5'    : results_dict['mAP@0.5'],
    'mAP@0.5:0.95': results_dict['mAP@0.5:0.95'],
    'Precision'  : results_dict['Precision'],
    'Recall'     : results_dict['Recall'],
    'FPS'        : results_dict.get('fps',0),
    'Params(M)'  : 25.9,
})

df = pd.DataFrame(compare_rows).sort_values('mAP@0.5', ascending=False)
print('\nBANG SO SANH BASELINE')
print(df.to_string(index=False))

df.to_csv(str(drive_root/'results'/'baseline_comparison.csv'), index=False)
print('\nDa luu: Drive/results/baseline_comparison.csv')

In [ ]:
# [4.4] Bieu do so sanh
import matplotlib.pyplot as plt

methods = [r['Phuong phap'].split('(')[0].strip() for r in compare_rows]
map50s  = [r['mAP@0.5']   for r in compare_rows]
fps_s   = [r['FPS']       for r in compare_rows]
colors  = ['#B5D4F4']*3 + ['#1D9E75']  # xanh cho baseline, xanh la cho de tai

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars1 = ax1.bar(methods, map50s, color=colors, edgecolor='white')
ax1.axhline(0.85, color='red', ls='--', lw=1.5, label='Muc tieu 0.85')
ax1.set_title('So sanh mAP@0.5', fontsize=12)
ax1.set_ylabel('mAP@0.5')
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=10)
for bar, v in zip(bars1, map50s):
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontsize=9)
ax1.tick_params(axis='x', rotation=15)

bars2 = ax2.bar(methods, fps_s, color=colors, edgecolor='white')
ax2.axhline(15, color='red', ls='--', lw=1.5, label='Muc tieu 15 FPS')
ax2.set_title('So sanh FPS', fontsize=12)
ax2.set_ylabel('FPS')
ax2.legend(fontsize=10)
for bar, v in zip(bars2, fps_s):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.0f}', ha='center', fontsize=9)
ax2.tick_params(axis='x', rotation=15)

plt.suptitle('Ket qua thuc nghiem so sanh baseline', fontsize=13, y=1.02)
plt.tight_layout()
save_p = str(drive_root/'results'/'baseline_chart.png')
plt.savefig(save_p, dpi=120, bbox_inches='tight')
plt.show()
print(f'Da luu: {save_p}')

In [ ]:
# [4.5] Export sang ONNX de tich hop vao pipeline
print('Export sang ONNX...')

exported = best_model.export(
    format   = 'onnx',
    imgsz    = CFG['img_size'],
    simplify = True,
    dynamic  = False,
    half     = False,
)
print(f'Export thanh cong: {exported}')

# Copy sang Drive
onnx_src = Path(exported)
onnx_dst = drive_root / 'checkpoints' / 'stage3_best.onnx'
shutil.copy2(onnx_src, onnx_dst)
print(f'ONNX saved to Drive: {onnx_dst}')

---
# Xu ly su co — Resume Training

> Dung cell nay khi Colab bi ngat giua chung.

In [ ]:
# [5.1] Resume tu checkpoint tren Drive
# Thay doi RESUME_STAGE thanh 2 hoac 3 tuy giai doan can resume
RESUME_STAGE = 3

ckpt_map = {
    2: str(drive_root / 'checkpoints' / 'stage2_best.pt'),
    3: str(drive_root / 'checkpoints' / 'stage3_last.pt'),  # last, khong phai best
}

ckpt_path = ckpt_map[RESUME_STAGE]
print(f'Resume Giai doan {RESUME_STAGE} tu: {ckpt_path}')
print(f'File ton tai: {Path(ckpt_path).exists()}')

if Path(ckpt_path).exists():
    model_resume = YOLO(ckpt_path)
    results_resume = model_resume.train(resume=True)
    print('Resume hoan tat!')
else:
    print('Khong tim thay checkpoint!')

---
# Tong ket ket qua

In [ ]:
# [6.1] In tong ket toan bo quy trinh
print('='*60)
print('TONG KET QUY TRINH TRAINING')
print('='*60)
print(f'De tai: Nhan dien phuong tien va uoc tinh mat do giao thong')
print(f'Sinh vien: Nguyen Huynh — 102220024')
print()
print('CHECKPOINT')
print(f'  Stage 1 (COCO)     : {CFG["base_model"]}')
print(f'  Stage 2 (BDD100K)  : Drive/checkpoints/stage2_best.pt')
print(f'  Stage 3 (VN final) : Drive/checkpoints/stage3_best.pt')
print(f'  ONNX export        : Drive/checkpoints/stage3_best.onnx')
print()
print('KET QUA CUOI')
try:
    with open(str(drive_root/'results'/'final_metrics.json')) as f:
        m = json.load(f)
    print(f'  mAP@0.5      : {m["mAP@0.5"]}')
    print(f'  mAP@0.5:0.95 : {m["mAP@0.5:0.95"]}')
    print(f'  Precision    : {m["Precision"]}')
    print(f'  Recall       : {m["Recall"]}')
    print(f'  FPS          : {m.get("fps", "chua do")}')
    print()
    print('Per-class mAP@0.5:')
    for cls, ap in m.get('per_class',{}).items():
        print(f'    {cls:8s}: {ap}')
except:
    print('  (Chua co metrics — chay cell [4.1] truoc)')
print()
print('OUTPUT FILES TREN DRIVE')
for f in (drive_root/'results').glob('*') if (drive_root/'results').exists() else []:
    print(f'  {f.name}')